# Phase 8: End-to-End Evaluation

**Pipeline**: Vietnamese Financial News RAG System — v3  
**Owner**: Member C | **Hardware**: Colab CPU

### Evaluation layers
| Layer | Method | API |
|-------|--------|-----|
| 1a | ROUGE-L (lexical overlap) | None |
| 1b | BERTScore (`bert-base-multilingual-cased`, `lang=vi`) | None |
| 2a | LLM-as-Judge: **Faithfulness** (1-5) | Groq |
| 2b | LLM-as-Judge: **Answer Relevance** (1-5) | Groq |

### Anti-crash mechanisms (re-used from Phase 7)
- **Global Cooldown**: `time.sleep(20)` between every pair of Groq calls  
- **Checkpoint**: parquet saved every 20 rows  
- **Idempotent resume**: completed rows detected and skipped on restart

> ⚠️ Do NOT run LLM-judge calls in a straight loop without cooldown — Groq free tier = 30 RPM.

## Cell 0 — Environment Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Runtime: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/thong7d/rag-vn-finance.git',
                        str(REPO_ROOT)])
        os.system(f'pip install -r "{REPO_ROOT}/requirements.txt" -q')
        os.kill(os.getpid(), 9)
    else:
        print("Repo already exists. Skipping install.")
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env')
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Not found: {REPO_ROOT}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Cell 0 complete.")

## Cell 1 — Imports, Config & API Key Check

In [ ]:
import time
import json
import pandas as pd
import numpy as np

from src.utils import load_config, resolve_path, ensure_dir, get_env, setup_logger
from src.evaluation import (
    compute_rouge_l,
    compute_bertscore,
    evaluate_faithfulness,
    evaluate_answer_relevance,
    GROQ_COOLDOWN,
    CHECKPOINT_EVERY,
)

logger = setup_logger("Phase8")
config = load_config()

# ── Paths ───────────────────────────────────────────────────────────────────
eval_dir   = resolve_path(config['evaluation'], 'output_dir')
ensure_dir(eval_dir)

gen_results_path   = os.path.join(eval_dir, 'generation_results.parquet')
eval_scores_path   = os.path.join(eval_dir, 'eval_scores.parquet')
final_table_path   = os.path.join(eval_dir, 'final_comparison_table.csv')

assert os.path.exists(gen_results_path), (
    f"generation_results.parquet not found at {gen_results_path}. "
    "Complete Phase 7 first."
)

# ── API key check ────────────────────────────────────────────────────────────
groq_key = get_env('GROQ_API_KEY')
assert groq_key, "GROQ_API_KEY not set. LLM-as-Judge cannot run."
print(f"✅ GROQ_API_KEY found — judge model: {config['evaluation']['groq_model']}")
print(f"   Global Cooldown : {GROQ_COOLDOWN}s between judge calls")
print(f"   Checkpoint every: {CHECKPOINT_EVERY} rows")

## Cell 2 — Load Phase 7 Generation Results

In [ ]:
df_gen = pd.read_parquet(gen_results_path)
print(f"Loaded {len(df_gen):,} generation results")
print(f"Columns: {df_gen.columns.tolist()}")

# Normalise column names (Phase 7 may use 'ground_truth' or 'reference_answer')
if 'ground_truth' in df_gen.columns:
    df_gen['reference_answer'] = df_gen['ground_truth']
elif 'reference_answer' not in df_gen.columns:
    raise KeyError("Neither 'ground_truth' nor 'reference_answer' column found.")

# Filter out rows where generation failed entirely
n_before = len(df_gen)
df_gen = df_gen[df_gen['generated_answer'].notna()].copy()
df_gen = df_gen[df_gen['generated_answer'] != '[GENERATION_ERROR]'].copy()
df_gen = df_gen.reset_index(drop=True)
print(f"Valid rows after removing errors: {len(df_gen)} / {n_before}")
df_gen.head(2)

## Cell 3 — Layer 1a: ROUGE-L

No API calls. Runs locally on CPU in ~30 seconds.

In [ ]:
from tqdm import tqdm

rouge_scores = []
for _, row in tqdm(df_gen.iterrows(), total=len(df_gen), desc="ROUGE-L"):
    score = compute_rouge_l(
        prediction=str(row['generated_answer']),
        reference=str(row['reference_answer']),
    )
    rouge_scores.append(score)

df_gen['rouge_l'] = rouge_scores
mean_rouge = np.mean(rouge_scores)
print(f"\nROUGE-L — mean: {mean_rouge:.4f}  "
      f"min: {min(rouge_scores):.4f}  max: {max(rouge_scores):.4f}")

## Cell 4 — Layer 1b: BERTScore

Uses `bert-base-multilingual-cased` with `lang="vi"` — mandatory for Vietnamese.  
Computes the full list in one batch call (~2–5 min on CPU).

In [ ]:
print("Computing BERTScore (bert-base-multilingual-cased, lang=vi)...")
print("This may take 2–5 minutes on CPU...")

bert_scores = compute_bertscore(
    predictions=df_gen['generated_answer'].astype(str).tolist(),
    references=df_gen['reference_answer'].astype(str).tolist(),
)

df_gen['bertscore_f1'] = bert_scores
mean_bert = np.mean(bert_scores)
print(f"\nBERTScore F1 — mean: {mean_bert:.4f}  "
      f"min: {min(bert_scores):.4f}  max: {max(bert_scores):.4f}")

## Cell 5 — Layer 2: LLM-as-Judge (Faithfulness + Answer Relevance)

**Anti-crash design** (mandatory):
- Each row makes exactly **2 Groq API calls** (Faithfulness, then Answer Relevance)
- A **`time.sleep(20)`** Global Cooldown fires between every pair of calls
- Results are **checkpointed every 20 rows** to `eval_scores.parquet`
- If the session crashes, re-running this cell resumes from the last checkpoint

> ⏱️ Expected time: ~197 rows × 40 s/row ≈ **~2.2 hours**. Leave the tab open.

In [ ]:
# ── Resume from checkpoint ────────────────────────────────────────────────────
if os.path.exists(eval_scores_path):
    df_scores_existing = pd.read_parquet(eval_scores_path)
    completed_indices  = set(df_scores_existing.index.tolist())
    # Use the stored original index to match df_gen rows
    if 'original_index' in df_scores_existing.columns:
        completed_indices = set(df_scores_existing['original_index'].tolist())
    all_score_rows = df_scores_existing.to_dict('records')
    print(f"🔄 Resuming — {len(all_score_rows)} rows already evaluated. Skipping them.")
else:
    completed_indices = set()
    all_score_rows    = []
    print("🚀 Starting fresh LLM-as-Judge evaluation...")

remaining = len(df_gen) - len(completed_indices)
print(f"   Rows to evaluate: {remaining}")
print(f"   Estimated time  : ~{remaining * (GROQ_COOLDOWN * 2 + 5) // 60} minutes\n")

# ── Evaluation loop (NOT a straight API loop — cooldown enforced per row) ─────
for orig_idx, row in tqdm(df_gen.iterrows(), total=len(df_gen), desc="LLM Judge"):

    # Skip rows already evaluated (idempotent resume)
    if orig_idx in completed_indices:
        continue

    question          = str(row['question'])
    generated_answer  = str(row['generated_answer'])
    retrieved_context = str(row.get('retrieved_context', ''))
    reference_answer  = str(row['reference_answer'])

    # ── Call 1: Faithfulness ─────────────────────────────────────────────────
    faith = evaluate_faithfulness(
        context=retrieved_context,
        answer=generated_answer,
    )

    # ── Global Cooldown (MANDATORY between consecutive Groq calls) ───────────
    logger.info(f"Row {orig_idx}: Faithfulness={faith['score']} — sleeping {GROQ_COOLDOWN}s")
    time.sleep(GROQ_COOLDOWN)

    # ── Call 2: Answer Relevance ─────────────────────────────────────────────
    relevance = evaluate_answer_relevance(
        question=question,
        answer=generated_answer,
    )
    logger.info(f"Row {orig_idx}: Relevance={relevance['score']}")

    # ── Accumulate result ────────────────────────────────────────────────────
    all_score_rows.append({
        'original_index':            orig_idx,
        'question':                  question,
        'generated_answer':          generated_answer,
        'reference_answer':          reference_answer,
        'rouge_l':                   row.get('rouge_l', 0.0),
        'bertscore_f1':              row.get('bertscore_f1', 0.0),
        'faithfulness_score':        faith.get('score', 0),
        'faithfulness_reasoning':    faith.get('reasoning', ''),
        'answer_relevance_score':    relevance.get('score', 0),
        'answer_relevance_reasoning':relevance.get('reasoning', ''),
        'strategy':                  row.get('strategy', ''),
        'method':                    row.get('method', ''),
        'doc_id':                    row.get('doc_id', ''),
    })

    # ── Checkpoint every CHECKPOINT_EVERY rows ───────────────────────────────
    if len(all_score_rows) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_score_rows).to_parquet(eval_scores_path, index=False)
        logger.info(f"[Checkpoint] {len(all_score_rows)} rows saved to {eval_scores_path}")

# Final save after loop completes
df_eval_scores = pd.DataFrame(all_score_rows)
df_eval_scores.to_parquet(eval_scores_path, index=False)
print(f"\n✅ LLM-as-Judge complete — {len(df_eval_scores)} rows evaluated.")
print(f"   Faithfulness    mean: {df_eval_scores['faithfulness_score'].mean():.2f}/5")
print(f"   Answer Relevance mean: {df_eval_scores['answer_relevance_score'].mean():.2f}/5")

## Cell 6 — Aggregate Metrics per Config (Strategy × Method)

In [ ]:
# Load the full scores table (handles case where session restarted after Cell 5)
df_scores = pd.read_parquet(eval_scores_path)
print(f"Loaded {len(df_scores)} scored rows")

# Numeric metric columns
METRIC_COLS = [
    'rouge_l', 'bertscore_f1',
    'faithfulness_score', 'answer_relevance_score',
]

# Group by Strategy × Method and compute mean
df_agg = (
    df_scores
    .groupby(['strategy', 'method'])[METRIC_COLS]
    .agg(['mean', 'std'])
    .round(4)
)

# Flatten multi-level column index
df_agg.columns = ['_'.join(c) for c in df_agg.columns]
df_agg = df_agg.reset_index()
print("\nPer-config aggregation:")
print(df_agg.to_string(index=False))

## Cell 7 — Merge with Retrieval Metrics & Save Final Comparison Table

In [ ]:
benchmark_path = os.path.join(eval_dir, 'retrieval_benchmark.csv')

if os.path.exists(benchmark_path):
    df_retrieval = pd.read_csv(benchmark_path)
    # Normalise column names to lowercase for merge
    df_retrieval = df_retrieval.rename(columns={'Strategy': 'strategy', 'Method': 'method'})
    df_final = df_agg.merge(df_retrieval, on=['strategy', 'method'], how='left')
    print("Merged with retrieval_benchmark.csv")
else:
    df_final = df_agg.copy()
    print("⚠️  retrieval_benchmark.csv not found — final table contains only generation metrics.")

# Save final comparison table
df_final.to_csv(final_table_path, index=False)
print(f"\n✅ Final comparison table saved: {final_table_path}")
print(f"   Rows: {len(df_final)}  Columns: {df_final.columns.tolist()}")
df_final

## Cell 8 — Best Configuration & Score Summary

In [ ]:
df_final = pd.read_csv(final_table_path)

# Rank by composite score: average of normalised MRR, ROUGE-L, BERTScore,
# Faithfulness, and Answer Relevance (all scaled 0-1)
score_cols_available = [c for c in [
    'MRR', 'rouge_l_mean', 'bertscore_f1_mean',
    'faithfulness_score_mean', 'answer_relevance_score_mean'
] if c in df_final.columns]

if score_cols_available:
    # Normalise each column to [0,1] then average
    df_norm = df_final[score_cols_available].copy()
    for col in score_cols_available:
        col_range = df_norm[col].max() - df_norm[col].min()
        df_norm[col] = (df_norm[col] - df_norm[col].min()) / (col_range if col_range > 0 else 1)
    df_final['composite_score'] = df_norm.mean(axis=1).round(4)
    best_idx = df_final['composite_score'].idxmax()
    best_row = df_final.loc[best_idx]
    print("=" * 60)
    print("BEST CONFIGURATION (by composite score)")
    print("=" * 60)
    print(f"  Strategy         : {best_row['strategy']}")
    print(f"  Method           : {best_row['method']}")
    print(f"  Composite Score  : {best_row['composite_score']:.4f}")
    for col in score_cols_available:
        print(f"  {col:30s}: {best_row[col]:.4f}")

print("\n" + "=" * 60)
print("ALL CONFIGURATIONS")
print("=" * 60)
print(df_final[['strategy', 'method'] + score_cols_available].to_string(index=False))

print(f"\n✅ Phase 8 complete. Confirm 'Xong' before proceeding to Phase 9.")

## Cell 9 — Error Analysis Sample

Display the 5 rows with the lowest Faithfulness score for manual inspection.

In [ ]:
df_scores = pd.read_parquet(eval_scores_path)

# Bottom 5 Faithfulness rows
bottom5 = df_scores.nsmallest(5, 'faithfulness_score')[
    ['question', 'generated_answer', 'faithfulness_score',
     'faithfulness_reasoning', 'answer_relevance_score']
]

print("=== 5 Lowest-Faithfulness Answers ===")
for _, row in bottom5.iterrows():
    print(f"\nFaithfulness={row['faithfulness_score']}  "
          f"Relevance={row['answer_relevance_score']}")
    print(f"Q  : {row['question'][:120]}")
    print(f"GEN: {str(row['generated_answer'])[:200]}")
    print(f"WHY: {row['faithfulness_reasoning']}")